In [1]:
import re
import pandas as pd
import requests
import math
from collections import defaultdict, Counter

<h4>Necessary imports: regular expressions,pandas, the math library requests, the defaultdict data structure and Counter</h4>

In [2]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

DOC_TITLES = [
    "Pizza",
    "The Hitchhiker's Guide to the Galaxy",
    "George Gershwin",
    "Paul Mccartney",
    "Cheese"
]
HEADERS = {
    "User-Agent": "Python/requests"
}

<h4>Constants defenition - Wikipedia's API, page titles and request necessaties</h4>

In [3]:
def wiki(title):
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": False,
        "titles": title,
        "format": "json",
        "redirects": 0,
        "formatversion": 2,
        "exintro" : 1
    }
    response = requests.get(WIKI_API, params=params, headers=HEADERS)
    pages = response.json().get("query", {}).get("pages", [])
    if not pages:
        return ""
    return pages[0].get("extract", "")

<h4>A function to handle and execute Wikipedia's API calls</h4>

In [4]:
def inputdataframe(titles):
    documents = {}
    countings = {}
    rows = []
    vocabulary = set()
    for title in titles:
        text = wiki(title) # fetch page
        tokens =[t for t in (re.split(r"\W+", text.lower())) if t] # tokenization
        count = Counter(tokens) # count token in page
        countings[title] = count #keep number of tokens for page
        vocabulary.update(count.keys())
    vocabulary = sorted(vocabulary)
    for title in titles:
        row = [countings[title].get(term,0) for term in vocabulary] # build rows for dataframe
        rows.append(row)
    df = pd.DataFrame(rows, index=titles, columns=vocabulary)
    return df

<h4>A function that builds a frequency dataframe containing the frequency of each term in each document in accordance to BoW principle</h4>

In [5]:
 inputdataframe(DOC_TITLES)

,000,1,100,11,128,13,18,1898,19,1919,...,world,would,writer,written,wrote,years,yesterday,york,you,zaphod
Pizza,1,0,0,0,1,1,0,0,0,0,...,2,0,0,0,0,1,0,0,0,0
The Hitchhiker's Guide to the Galaxy,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,1
George Gershwin,0,0,0,1,0,0,0,1,0,1,...,0,1,0,0,1,1,0,1,1,0
Paul Mccartney,0,1,2,0,0,0,1,0,1,0,...,1,0,0,2,2,0,1,0,0,0
Cheese,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
def getmetrics(df, k1=1.6, b=0.75):
    N = df.shape[0]
    doc_lens = df.sum(axis=1).astype(float)                # get length per doc
    avgdl = doc_lens.mean()
    df_term = (df > 0).sum(axis=0).to_dict() # calculate document frequency for each term 
    idf = {term: math.log((N) / (df_term[term])) for term in df.columns} 
    return {"df": df_term, "idf": idf, "doc_lens": doc_lens, "avgdl": avgdl, "k1": k1, "b": b, "dataframe": df}

<h4>A function that retrieves the necessary metrics from the corpora to calculate Okapi BM25 from the input dataframe</h4>

In [7]:
def bm25dataframe(metrics, query_terms):
    df, idf, k1, b, avgdl, doc_lens = metrics["dataframe"], metrics["idf"], metrics["k1"], metrics["b"], metrics["avgdl"], metrics["doc_lens"]
    denomenator_base = k1 * (1 - b + b * (doc_lens / avgdl))
    contribution = pd.DataFrame(0.0, index=df.index, columns=query_terms)
    for term in query_terms:
        if term in df.columns:
            tf = df[term].astype(float)
            denomenator = tf + denomenator_base
            contribution[term] = idf.get(term, 0.0) * (tf * (k1 + 1) / denomenator)
        else:
            contribution[term] = 0.0
    contribution["Okapi BM25"] = contribution.sum(axis=1)
    cols = ["Okapi BM25"] + list(query_terms)
    return contribution[cols]


<h4>A function that iteratively calculates Okapi BM25 for the document collection and a query, in accordance with the below formula</h4>

$
\mathrm{Okapi\ BM25}(D,|Q|)=\sum_{i=1}^{|Q|}
\mathrm{IDF_i}\cdot
\frac{TF_i\cdot(k_1+1)}
{TF_i+k_1\cdot\left(1-b+b\cdot\frac{dl}{\mathrm{avgdl}}\right)}
\
$

<h4>where D is the corpora of documents and Q is the query</h4>

In [8]:
bm25dataframe(getmetrics(inputdataframe(DOC_TITLES)), input('input terms to search: ').split())

input terms to search:  british music is well recieved


,Okapi BM25,british,music,is,well,recieved
Pizza,0.490605,0.000000,0.000000,0.490605,0.0,0.0
The Hitchhiker's Guide to the Galaxy,0.396641,0.000000,0.000000,0.396641,0.0,0.0
George Gershwin,1.036127,0.000000,1.036127,0.000000,0.0,0.0
Paul Mccartney,3.275062,1.255243,1.624264,0.395556,0.0,0.0
Cheese,0.452027,0.000000,0.000000,0.452027,0.0,0.0
